# Descrição

Este notebook anota os Ensembl Gene IDs com símbolos gênicos usando `mygene`.

A anotação será usada para integrar os resultados de expressão diferencial com os módulos de WGCNA e, posteriormente, preparar listas de genes para análise no STRING e no Cytoscape.

## Imports

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import mygene

## Configurações

In [2]:
PROCESSED_DIR = Path("../../data/interim")
DESEQ_DIR = PROCESSED_DIR / "deseq2"
WGCNA_DIR = PROCESSED_DIR / "wgcna"
ANNOTATION_DIR = PROCESSED_DIR / "annotation"

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

# Entradas
DEG_ALL_CONTRASTS_PATH = DESEQ_DIR / "deg_all_contrasts.csv"
WGCNA_GENE_MODULES_PATH = WGCNA_DIR / "wgcna_gene_modules.csv"

# Saídas
GENE_ANNOTATION_PATH = ANNOTATION_DIR / "gene_annotation.csv"
DEG_ALL_ANNOTATED_PATH = ANNOTATION_DIR / "deg_all_contrasts_annotated.csv"
WGCNA_MODULES_ANNOTATED_PATH = ANNOTATION_DIR / "wgcna_gene_modules_annotated.csv"

## Carregamento dos Dados

In [3]:
deg_df = pd.read_csv(DEG_ALL_CONTRASTS_PATH)
wgcna_df = pd.read_csv(WGCNA_GENE_MODULES_PATH)

print("DEG:", deg_df.shape)
print("WGCNA modules:", wgcna_df.shape)

display(deg_df.head())
display(wgcna_df.head())

DEG: (121740, 10)
WGCNA modules: (12174, 3)


,gene_id,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,contrast,significant,direction
0,ENSG00000185112,1161.674476,1.476968,0.188468,7.836698,4.625473e-15,4.102332e-11,MA100_vs_CTR,True,up_in_MA100
1,ENSG00000135046,4815.301901,-0.970120,0.136702,-7.096592,1.278713e-12,5.138223e-09,MA100_vs_CTR,True,up_in_CTR
2,ENSG00000040275,2053.578939,1.214744,0.172206,7.054034,1.738039e-12,5.138223e-09,MA100_vs_CTR,True,up_in_MA100
3,ENSG00000214357,1332.742635,1.313576,0.196726,6.677189,2.435686e-11,5.400524e-08,MA100_vs_CTR,True,up_in_MA100
4,ENSG00000147133,1810.470521,1.281281,0.192941,6.640789,3.120084e-11,5.534405e-08,MA100_vs_CTR,True,up_in_MA100


,gene_id,dynamic_module,module
0,ENSG00000000003,dimgrey,dimgrey
1,ENSG00000000419,dimgrey,dimgrey
2,ENSG00000000457,dimgrey,dimgrey
3,ENSG00000000460,dimgrey,dimgrey
4,ENSG00000001036,dimgrey,dimgrey


## Anotação dos Dados

In [4]:
# Coleta de genes únicos para anotação

def clean_ensembl_id(gene_id):
    """
    Remove versão de Ensembl ID, caso exista.
    Exemplo: ENSG00000123456.10 -> ENSG00000123456
    """
    if pd.isna(gene_id):
        return np.nan
    return str(gene_id).split(".")[0]


deg_df["gene_id"] = deg_df["gene_id"].map(clean_ensembl_id)
wgcna_df["gene_id"] = wgcna_df["gene_id"].map(clean_ensembl_id)

gene_ids = sorted(
    set(deg_df["gene_id"].dropna()) |
    set(wgcna_df["gene_id"].dropna())
)

print("Genes únicos para anotação:", len(gene_ids))
print(gene_ids[:10])

Genes únicos para anotação: 12174
['ENSG00000000003', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460', 'ENSG00000001036', 'ENSG00000001084', 'ENSG00000001167', 'ENSG00000001460', 'ENSG00000001461', 'ENSG00000001497']


In [5]:
# Consulta ao MyGene para anotação

mg = mygene.MyGeneInfo()

if GENE_ANNOTATION_PATH.exists():
    print(f"Carregando anotações de {GENE_ANNOTATION_PATH}...")
    annotation_df = pd.read_csv(GENE_ANNOTATION_PATH)
    
    annotation_df["gene_id"] = annotation_df["gene_id"].map(clean_ensembl_id)

    if "notfound" not in annotation_df.columns:
        annotation_df["notfound"] = annotation_df["gene_symbol"].isna()
else:
    results = mg.querymany(
        gene_ids,
        scopes="ensembl.gene",
        fields="symbol,name",
        species="human",
        as_dataframe=False
    )

    annotation_rows = []

    for item in results:
        gene_id = clean_ensembl_id(item.get("query"))
        
        annotation_rows.append({
            "gene_id": gene_id,
            "gene_symbol": item.get("symbol", np.nan),
            "gene_name": item.get("name", np.nan),
            "notfound": item.get("notfound", False),
        })

    annotation_df = pd.DataFrame(annotation_rows)

    annotation_df = (
        annotation_df
        .sort_values(["gene_id", "notfound"])
        .drop_duplicates(subset="gene_id", keep="first")
        .reset_index(drop=True)
    )

    annotation_df.to_csv(GENE_ANNOTATION_PATH, index=False)
    print(f"Anotação salva em: {GENE_ANNOTATION_PATH}")

annotation_df = (
    annotation_df
    .sort_values(["gene_id", "notfound"])
    .drop_duplicates(subset="gene_id", keep="first")
    .reset_index(drop=True)
)

print("Anotações:", annotation_df.shape)
display(annotation_df.head())

Carregando anotações de ../../data/interim/annotation/gene_annotation.csv...
Anotações: (12174, 5)


,gene_id,gene_symbol,gene_name,notfound,gene_label
0,ENSG00000000003,TSPAN6,tetraspanin 6,False,TSPAN6
1,ENSG00000000419,DPM1,dolichyl-phosphate mannosyltransferase subunit...,False,DPM1
2,ENSG00000000457,SCYL3,SCY1 like pseudokinase 3,False,SCYL3
3,ENSG00000000460,FIRRM,FIGNL1 interacting regulator of recombination ...,False,FIRRM
4,ENSG00000001036,FUCA2,alpha-L-fucosidase 2,False,FUCA2


In [6]:
# Verificação do resultado da anotação

n_total = annotation_df.shape[0]
n_mapped = annotation_df["gene_symbol"].notna().sum()
n_unmapped = annotation_df["gene_symbol"].isna().sum()

print(f"Total de genes: {n_total}")
print(f"Genes anotados com symbol: {n_mapped}")
print(f"Genes sem symbol: {n_unmapped}")
print(f"Taxa de anotação: {n_mapped / n_total:.2%}")

display(
    annotation_df[annotation_df["gene_symbol"].isna()].head(20)
)

Total de genes: 12174
Genes anotados com symbol: 12174
Genes sem symbol: 0
Taxa de anotação: 100.00%


,gene_id,gene_symbol,gene_name,notfound,gene_label


In [7]:
# Salvamento da anotação

annotation_df.to_csv(GENE_ANNOTATION_PATH, index=False)

print("Tabela de anotação salva em:")
print(GENE_ANNOTATION_PATH)

Tabela de anotação salva em:
../../data/interim/annotation/gene_annotation.csv


## Anotação da Tabela DEG

In [8]:
deg_annotated_df = deg_df.merge(
    annotation_df[["gene_id", "gene_symbol", "gene_name"]],
    on="gene_id",
    how="left"
)

# Reorganiza colunas principais
front_cols = [
    "gene_id",
    "gene_symbol",
    "gene_name",
    "contrast",
    "baseMean",
    "log2FoldChange",
    "lfcSE",
    "stat",
    "pvalue",
    "padj",
    "significant",
    "direction",
]

front_cols = [col for col in front_cols if col in deg_annotated_df.columns]
other_cols = [col for col in deg_annotated_df.columns if col not in front_cols]

deg_annotated_df = deg_annotated_df[front_cols + other_cols]

deg_annotated_df.to_csv(DEG_ALL_ANNOTATED_PATH, index=False)

print("DEG anotado salvo em:")
print(DEG_ALL_ANNOTATED_PATH)
print("Dimensões:", deg_annotated_df.shape)

display(deg_annotated_df.head())

DEG anotado salvo em:
../../data/interim/annotation/deg_all_contrasts_annotated.csv
Dimensões: (121740, 12)


,gene_id,gene_symbol,gene_name,contrast,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,significant,direction
0,ENSG00000185112,FAM43A,family with sequence similarity 43 member A,MA100_vs_CTR,1161.674476,1.476968,0.188468,7.836698,4.625473e-15,4.102332e-11,True,up_in_MA100
1,ENSG00000135046,ANXA1,annexin A1,MA100_vs_CTR,4815.301901,-0.970120,0.136702,-7.096592,1.278713e-12,5.138223e-09,True,up_in_CTR
2,ENSG00000040275,SPDL1,spindle apparatus coiled-coil protein 1,MA100_vs_CTR,2053.578939,1.214744,0.172206,7.054034,1.738039e-12,5.138223e-09,True,up_in_MA100
3,ENSG00000214357,NEURL1B,neuralized E3 ubiquitin protein ligase 1B,MA100_vs_CTR,1332.742635,1.313576,0.196726,6.677189,2.435686e-11,5.400524e-08,True,up_in_MA100
4,ENSG00000147133,TAF1,TATA-box binding protein associated factor 1,MA100_vs_CTR,1810.470521,1.281281,0.192941,6.640789,3.120084e-11,5.534405e-08,True,up_in_MA100


In [9]:
# Verificação do resultado da anotação

n_total = deg_annotated_df.shape[0]
n_mapped = deg_annotated_df["gene_symbol"].notna().sum()
n_unmapped = deg_annotated_df["gene_symbol"].isna().sum()

print(f"Total de genes: {n_total}")
print(f"Genes anotados com symbol: {n_mapped}")
print(f"Genes sem symbol: {n_unmapped}")
print(f"Taxa de anotação: {n_mapped / n_total:.2%}")

display(
    deg_annotated_df[deg_annotated_df["gene_symbol"].isna()].head(20)
)

Total de genes: 121740
Genes anotados com symbol: 121740
Genes sem symbol: 0
Taxa de anotação: 100.00%


,gene_id,gene_symbol,gene_name,contrast,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,significant,direction


## Anotação da Tabela WGCNA

In [10]:
wgcna_annotated_df = wgcna_df.merge(
    annotation_df[["gene_id", "gene_symbol", "gene_name"]],
    on="gene_id",
    how="left"
)

front_cols = [
    "gene_id",
    "gene_symbol",
    "gene_name",
    "dynamic_module",
    "module",
]

front_cols = [col for col in front_cols if col in wgcna_annotated_df.columns]
other_cols = [col for col in wgcna_annotated_df.columns if col not in front_cols]

wgcna_annotated_df = wgcna_annotated_df[front_cols + other_cols]

wgcna_annotated_df.to_csv(WGCNA_MODULES_ANNOTATED_PATH, index=False)

print("WGCNA anotado salvo em:")
print(WGCNA_MODULES_ANNOTATED_PATH)
print("Dimensões:", wgcna_annotated_df.shape)

display(wgcna_annotated_df.head())

WGCNA anotado salvo em:
../../data/interim/annotation/wgcna_gene_modules_annotated.csv
Dimensões: (12174, 5)


,gene_id,gene_symbol,gene_name,dynamic_module,module
0,ENSG00000000003,TSPAN6,tetraspanin 6,dimgrey,dimgrey
1,ENSG00000000419,DPM1,dolichyl-phosphate mannosyltransferase subunit...,dimgrey,dimgrey
2,ENSG00000000457,SCYL3,SCY1 like pseudokinase 3,dimgrey,dimgrey
3,ENSG00000000460,FIRRM,FIGNL1 interacting regulator of recombination ...,dimgrey,dimgrey
4,ENSG00000001036,FUCA2,alpha-L-fucosidase 2,dimgrey,dimgrey


In [11]:
# Verificação do resultado da anotação

n_total = wgcna_annotated_df.shape[0]
n_mapped = wgcna_annotated_df["gene_symbol"].notna().sum()
n_unmapped = wgcna_annotated_df["gene_symbol"].isna().sum()

print(f"Total de genes: {n_total}")
print(f"Genes anotados com symbol: {n_mapped}")
print(f"Genes sem symbol: {n_unmapped}")
print(f"Taxa de anotação: {n_mapped / n_total:.2%}")

display(
    wgcna_annotated_df[wgcna_annotated_df["gene_symbol"].isna()].head(20)
)

Total de genes: 12174
Genes anotados com symbol: 12174
Genes sem symbol: 0
Taxa de anotação: 100.00%


,gene_id,gene_symbol,gene_name,dynamic_module,module
